# Week 2: Text Cleaning & Normalization

This week focused on profiling and cleaning the MLS remarks column. The goal was to standardize common abbreviations, prices, measurements, unicode characters, HTML remnants, punctuation, and whitespace so the text is more consistent for future NLP tasks.

In [5]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath(".."))

from scripts.text_cleaning import TextCleaner

df = pd.read_csv("../data/processed/listing_sample.csv")

cleaner = TextCleaner()

df.head()

,L_ListingID,L_Address,L_City,beds,baths,price,remarks
0,1190668831,1220 Johnson Drive 107,Ventura,2.0,2.0,650000,This is the special Gated 55+ Rancho Ventura h...
1,1174095292,850 Beech Street 511,San Diego,3.0,2.0,839900,"Experience uncompromised peace of mind, ultima..."
2,1190575810,141 Mellano Way,Oceanside,4.0,5.0,1299900,This two-story Next Gen® home that now include...
3,1189565343,2248 1/2 Gatewood Street,Los Angeles,2.0,3.0,665000,Thoughtful architecture meets iconic Eastside ...
4,1178725764,503 N Via Belmonte Court,Arroyo Grande,3.0,3.0,1999995,An extraordinary opportunity to own one of Arr...


In [6]:
print("Null count:", df["remarks"].isnull().sum())
print("Null rate:", df["remarks"].isnull().mean())
print("Average length:", df["remarks"].str.len().mean())
print("Median length:", df["remarks"].str.len().median())

Null count: 0
Null rate: 0.0
Average length: 1306.967
Median length: 1250.5


In [7]:
html_count = df["remarks"].str.contains(
    r"<[^>]+>",
    regex=True,
    na=False
).sum()

print("Rows with HTML:", html_count)

Rows with HTML: 0


In [8]:
price_count = df["remarks"].str.contains(
    r"\$\s*\d",
    regex=True,
    na=False
).sum()

print("Rows with price mentions:", price_count)

Rows with price mentions: 73


In [9]:
abbreviations = [
    "br", "bdr", "bdrm", "bd",
    "ba", "bath", "bth",
    "sqft", "sq ft", "sf",
    "w/", "w/o",
    "mbr", "mbdr",
    "lr", "dr", "fr",
    "kit", "gar",
    "ac", "a/c",
    "hw", "hdwd",
    "bsmt", "bsmnt",
    "apt", "condo",
    "pkg", "prkg",
    "lau", "lndry",
    "flr", "lvl",
    "yr", "yrs",
    "mo", "mins", "min",
    "approx", "incl", "excl"
]

In [10]:
import re

abbrev_counts = {}

for abbrev in abbreviations:
    pattern = r"(?<!\w)" + re.escape(abbrev) + r"(?!\w)"
    
    count = df["remarks"].str.count(
        pattern,
        flags=re.IGNORECASE
    ).sum()
    
    if count > 0:
        abbrev_counts[abbrev] = int(count)

abbrev_counts

{'br': 5,
 'bd': 3,
 'ba': 8,
 'bath': 536,
 'sqft': 34,
 'sq ft': 83,
 'sf': 37,
 'w/': 16,
 'dr': 10,
 'fr': 1,
 'ac': 32,
 'a/c': 59,
 'hw': 1,
 'condo': 139,
 'yrs': 1,
 'mins': 2,
 'min': 1,
 'approx': 20,
 'incl': 5}

## Data Profiling Findings

The remarks column contained no missing values and no HTML tags. The average remark length was about 1,307 characters, with a median length of about 1,251 characters. Price mentions appeared in 73 listings.

Several common real estate abbreviations were identified, including `bath`, `condo`, `sq ft`, `a/c`, `sf`, `sqft`, `ac`, `approx`, and `w/`. These findings helped determine which normalization rules were most relevant for the dataset.

In [12]:
bath_examples = df[
    df["remarks"].str.contains(
        r"\bbath\b",
        case=False,
        na=False
    )
]["remarks"].head(10)

for i, text in bath_examples.items():
    print(f"\nROW {i}:")
    print(text)
    print("-" * 80)


ROW 0:
This is the special Gated 55+ Rancho Ventura home you've been waiting for! Spacious and upgraded 1680sf of easy living on a large corner lot. Two generous bedrooms, plus a separate office/bonus room with a solar tube could be used for guests. There are two large sunny living spaces, and the family room has a lovely fireplace with a raised hearth.Tasteful upgrades include the granite kitchen with stainless appliances, the primary bath has been beautifully remodeled, all new windows, newer heating and central air-conditioning, lighting and ceiling fans, and gorgeous hardwood floors. Your outdoor space has a charming gazebo on the patio and a large side yard. Even the garage has new epoxy floors! The low $365 HOA fee includes water, sewer, trash, basic Spectrum cable/internet, and the use of all of Rancho Ventura's fabulous amenities: clubhouse, heated pool, spa, kitchen, library, weight room, billiards room, RV/Trailer parking and the friendliness neighbors in Ventura! Rancho Ven

In [13]:
cleaner = TextCleaner()

cleaner.expand_abbreviations("3 br 2 ba home w/ ac")


'3 bedroom 2 bathroom home with air conditioning'

In [14]:
cleaner.normalize_prices("priced at 450k")

'priced at 450000'

In [15]:
cleaner.normalize_prices("$1.2m home")

'$1200000 home'

In [16]:
cleaner.normalize_prices("Listed at 750K")

'Listed at 750000'

In [17]:
profile = cleaner.profile_column(df, "remarks")

profile

{'null_rate': 0.0,
 'null_count': 0,
 'avg_length': 1306.967,
 'median_length': 1250.5,
 'price_mentions': 73,
 'has_html': 0,
 'unicode_issues': 355,
 'common_terms': [('and', 9358),
  ('the', 8582),
  ('a', 7858),
  ('with', 3623),
  ('to', 3358),
  ('of', 2896),
  ('in', 2881),
  ('for', 2540),
  ('home', 2461),
  ('this', 2218),
  ('living', 1853),
  ('is', 1752),
  ('an', 1639),
  ('room', 1318),
  ('space', 1153)],
 'common_abbreviations': [('bath', 536),
  ('condo', 139),
  ('sq ft', 83),
  ('a/c', 60),
  ('w/', 49),
  ('sf', 37),
  ('sqft', 34),
  ('ac', 32),
  ('approx', 20),
  ('dr', 10),
  ('ba', 8),
  ('br', 5),
  ('incl', 5),
  ('bd', 3),
  ('rec', 2),
  ('mins', 2),
  ('yrs', 1),
  ('fr', 1),
  ('min', 1),
  ('hw', 1)]}

### Abbreviation Validation

Some terms were checked manually before being added to the normalization rules because they could be ambiguous. For example, listings containing the word `bath` were reviewed to confirm that it was being used to refer to a bathroom in this dataset.

In [18]:
terms_df = pd.DataFrame(
    profile["common_terms"],
    columns=["term", "count"]
)

terms_df

,term,count
0,and,9358
1,the,8582
2,a,7858
3,with,3623
4,to,3358
5,of,2896
6,in,2881
7,for,2540
8,home,2461
9,this,2218


In [19]:
abbrev_df = pd.DataFrame(
    profile["common_abbreviations"],
    columns=["abbreviation", "count"]
)

abbrev_df.head(20)

,abbreviation,count
0,bath,536
1,condo,139
2,sq ft,83
3,a/c,60
4,w/,49
5,sf,37
6,sqft,34
7,ac,32
8,approx,20
9,dr,10


## Text Cleaning Pipeline

The `TextCleaner` class applies several normalization steps:

- Unicode normalization
- HTML removal
- Price normalization
- Measurement normalization
- Abbreviation expansion
- Punctuation normalization
- Whitespace normalization

The complete pipeline is applied to each MLS remark to create a new `remarks_clean` column while preserving the original remarks for comparison.

In [20]:
df["remarks_clean"] = df["remarks"].apply(cleaner.clean_text)

df.head()

,L_ListingID,L_Address,L_City,beds,baths,price,remarks,remarks_clean
0,1190668831,1220 Johnson Drive 107,Ventura,2.0,2.0,650000,This is the special Gated 55+ Rancho Ventura h...,This is the special Gated 55+ Rancho Ventura h...
1,1174095292,850 Beech Street 511,San Diego,3.0,2.0,839900,"Experience uncompromised peace of mind, ultima...","Experience uncompromised peace of mind, ultima..."
2,1190575810,141 Mellano Way,Oceanside,4.0,5.0,1299900,This two-story Next Gen® home that now include...,This two-story Next Gen® home that now include...
3,1189565343,2248 1/2 Gatewood Street,Los Angeles,2.0,3.0,665000,Thoughtful architecture meets iconic Eastside ...,Thoughtful architecture meets iconic Eastside ...
4,1178725764,503 N Via Belmonte Court,Arroyo Grande,3.0,3.0,1999995,An extraordinary opportunity to own one of Arr...,An extraordinary opportunity to own one of Arr...


In [22]:
df.to_csv(
    "../data/processed/week2_cleaned.csv",
    index=False
)

In [23]:
changed = df[
    df["remarks"] != df["remarks_clean"]
][["remarks", "remarks_clean"]]

changed.head(10)

,remarks,remarks_clean
0,This is the special Gated 55+ Rancho Ventura h...,This is the special Gated 55+ Rancho Ventura h...
1,"Experience uncompromised peace of mind, ultima...","Experience uncompromised peace of mind, ultima..."
2,This two-story Next Gen® home that now include...,This two-story Next Gen® home that now include...
3,Thoughtful architecture meets iconic Eastside ...,Thoughtful architecture meets iconic Eastside ...
5,Location! Location! Great Location! This char...,Location! Location! Great Location! This charm...
6,"This beautiful one bedroom, one bathroom home ...","This beautiful one bedroom, one bathroom home ..."
7,Modern Living with Room to Breathe.\r\r\nEscap...,Modern Living with Room to Breathe. Escape to ...
8,Welcome to your detached single family dream h...,Welcome to your detached single family dream h...
10,Meet this Historic Grandeur with Modern Flexib...,Meet this Historic Grandeur with Modern Flexib...
11,"Rare, highly sought after NE Corner live/work ...","Rare, highly sought after NE Corner live/work ..."


In [24]:
pd.set_option("display.max_colwidth", 200)

changed.head(5)

,remarks,remarks_clean
0,"This is the special Gated 55+ Rancho Ventura home you've been waiting for! Spacious and upgraded 1680sf of easy living on a large corner lot. Two generous bedrooms, plus a separate office/bonus ro...","This is the special Gated 55+ Rancho Ventura home you've been waiting for! Spacious and upgraded 1680 square feet of easy living on a large corner lot. Two generous bedrooms, plus a separate offic..."
1,"Experience uncompromised peace of mind, ultimate privacy, and resort-style living at The Discovery in Cortez Hill. High-rise living in downtown San Diego is at its absolute finest when backed by a...","Experience uncompromised peace of mind, ultimate privacy, and resort-style living at The Discovery in Cortez Hill. High-rise living in downtown San Diego is at its absolute finest when backed by a..."
2,"This two-story Next Gen® home that now includes backsplashes and upgraded flooring and is ready for a Quick Move-In, features a private suite with a separate entrance, living area, kitchenette and...","This two-story Next Gen® home that now includes backsplashes and upgraded flooring and is ready for a Quick Move-In, features a private suite with a separate entrance, living area, kitchenette and..."
3,"Thoughtful architecture meets iconic Eastside energy in one of Los Angeles' most creative and rapidly evolving neighborhoods. Be the first to live in this ground up new construction 2 bed, 2.5 bat...","Thoughtful architecture meets iconic Eastside energy in one of Los Angeles' most creative and rapidly evolving neighborhoods. Be the first to live in this ground up new construction 2 bed, 2.5 bat..."
5,"Location! Location! Great Location! This charming Greek-inspired, blue-and-white ranch-style home is tucked away on a peaceful, sought-after cul-de-sac in the heart of Midtown Palo Alto. The home...","Location! Location! Great Location! This charming Greek-inspired, blue-and-white ranch-style home is tucked away on a peaceful, sought-after cul-de-sac in the heart of Midtown Palo Alto. The home ..."


In [25]:
changed_count = (
    df["remarks"] != df["remarks_clean"]
).sum()

changed_rate = changed_count / len(df)

print(f"Remarks changed: {changed_count:,}")
print(f"Percentage changed: {changed_rate:.2%}")

Remarks changed: 863
Percentage changed: 86.30%


In [26]:
df["original_length"] = df["remarks"].str.len()

df["clean_length"] = df["remarks_clean"].str.len()

print(
    "Average original length:",
    df["original_length"].mean()
)

print(
    "Average cleaned length:",
    df["clean_length"].mean()
)

Average original length: 1306.967
Average cleaned length: 1308.25


## Results

The cleaned remarks were compared with the original remarks to determine how frequently the pipeline changed the text. Before-and-after examples were also reviewed to make sure the normalization rules improved consistency without changing the intended meaning.

The cleaned dataset was saved as `data/processed/week2_cleaned.csv` for use in later stages of the project.
